# CineMagic — 豆瓣电影数据探索分析 (EDA)

本 notebook 对豆瓣 Top200 电影数据集进行探索性数据分析，包括：
- 评分分布
- 用户活跃度
- 电影流行度
- 类型分析
- 稀疏度分析
- 用户-电影交互热力图

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# 中文支持
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.bbox'] = 'tight'

from src.data.loader import load_movies, load_ratings, load_top200_data
from src.data.preprocess import filter_sparse, build_user_item_matrix

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# 加载数据
MOVIES_PATH = '../data/douban/top200.csv'
RATINGS_PATH = '../data/douban/top200_ratings.csv'

movies = load_movies(MOVIES_PATH)
ratings = load_ratings(RATINGS_PATH)

print(f'Movies: {len(movies)}')
print(f'Ratings: {len(ratings)}')
print(f'Users: {ratings["userId"].nunique()}')
print(f'Sparsity: {len(ratings) / (ratings["userId"].nunique() * len(movies)) * 100:.4f}%')

In [ ]:
movies.head(3)

In [ ]:
ratings.head(3)

## 1. 评分分布

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 用户评分分布
ax = axes[0]
rating_counts = ratings['rating'].value_counts().sort_index()
bars = ax.bar(rating_counts.index, rating_counts.values, color='steelblue', edgecolor='white')
ax.set_title('User Rating Distribution', fontsize=14)
ax.set_xlabel('Rating')
ax.set_ylabel('Count')
for bar, count in zip(bars, rating_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500, f'{count:,}', ha='center', fontsize=9)

# 豆瓣评分分布
ax = axes[1]
douban_scores = movies['DOUBAN_SCORE'].replace(0.0, np.nan).dropna()
ax.hist(douban_scores, bins=25, color='coral', edgecolor='white')
ax.set_title('Douban Score Distribution', fontsize=14)
ax.set_xlabel('Douban Score')
ax.set_ylabel('Count')
ax.axvline(douban_scores.median(), color='red', linestyle='--', label=f'Median={douban_scores.median():.1f}')
ax.legend()

# 豆瓣评分数分布
ax = axes[2]
votes = movies['DOUBAN_VOTES'].replace(0.0, np.nan).dropna()
ax.hist(np.log10(votes), bins=25, color='seagreen', edgecolor='white')
ax.set_title('Douban Votes Distribution (log10)', fontsize=14)
ax.set_xlabel('log10(Votes)')
ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# 评分统计
print(f'User ratings:  mean={ratings["rating"].mean():.2f}, std={ratings["rating"].std():.2f}')
print(f'Douban scores: mean={douban_scores.mean():.2f}, median={douban_scores.median():.2f}')
print(f'Douban votes:  mean={votes.mean():.0f}, median={votes.median():.0f}')

## 2. 用户活跃度分析

In [ ]:
user_stats = ratings.groupby('userId').agg(
    rating_count=('rating', 'count'),
    rating_mean=('rating', 'mean'),
    rating_std=('rating', 'std'),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 用户评分数分布
ax = axes[0]
ax.hist(user_stats['rating_count'].clip(upper=50), bins=50, color='steelblue', edgecolor='white')
ax.set_title(f'Ratings per User (median={user_stats["rating_count"].median():.0f})', fontsize=14)
ax.set_xlabel('Number of Ratings (clipped at 50)')
ax.set_ylabel('Number of Users')

# 用户评分均值分布
ax = axes[1]
ax.hist(user_stats['rating_mean'], bins=30, color='coral', edgecolor='white')
ax.set_title('User Mean Rating Distribution', fontsize=14)
ax.set_xlabel('Mean Rating')
ax.axvline(user_stats['rating_mean'].median(), color='red', linestyle='--')

# 评分数 vs 评分均值
ax = axes[2]
ax.scatter(
    user_stats['rating_count'].clip(upper=100),
    user_stats['rating_mean'],
    alpha=0.3, s=5, color='purple'
)
ax.set_title('Rating Count vs Mean Rating', fontsize=14)
ax.set_xlabel('Number of Ratings')
ax.set_ylabel('Mean Rating')

plt.tight_layout()
plt.show()

print(f'Total users: {len(user_stats):,}')
print(f'Users with >= 5 ratings: {(user_stats["rating_count"] >= 5).sum():,}')
print(f'Users with >= 10 ratings: {(user_stats["rating_count"] >= 10).sum():,}')
print(f'Users with 1 rating only: {(user_stats["rating_count"] == 1).sum():,} ({(user_stats["rating_count"] == 1).sum()/len(user_stats)*100:.1f}%)')

## 3. 电影流行度分析

In [ ]:
movie_stats = ratings.groupby('movieId').agg(
    user_count=('rating', 'count'),
    avg_rating=('rating', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 每部电影评分数分布
ax = axes[0]
ax.hist(movie_stats['user_count'], bins=40, color='steelblue', edgecolor='white')
ax.set_title(f'Ratings per Movie (median={movie_stats["user_count"].median():.0f})', fontsize=14)
ax.set_xlabel('Number of Ratings')
ax.set_ylabel('Number of Movies')

# 平均评分分布
ax = axes[1]
ax.hist(movie_stats['avg_rating'], bins=30, color='coral', edgecolor='white')
ax.set_title('Movie Average User Rating', fontsize=14)
ax.set_xlabel('Average Rating')

# 流行度长尾 (Log-Log)
ax = axes[2]
sorted_counts = movie_stats['user_count'].sort_values(ascending=False).values
ax.loglog(range(1, len(sorted_counts)+1), sorted_counts, 'b-', linewidth=1)
ax.set_title('Popularity Long Tail (Log-Log)', fontsize=14)
ax.set_xlabel('Movie Rank (by popularity)')
ax.set_ylabel('Number of Ratings')

plt.tight_layout()
plt.show()

print(f'Movies: {len(movie_stats)}')
print(f'Top 10 most rated movies:')
top10 = movie_stats.nlargest(10, 'user_count')
for _, row in top10.iterrows():
    movie_id = row['movieId']
    name = movies.at[movie_id, 'NAME'] if movie_id in movies.index else 'Unknown'
    print(f'  {name[:40]:<42} ratings={row["user_count"]:>5}  avg={row["avg_rating"]:.2f}')

## 4. 电影类型分析

In [ ]:
from collections import Counter

# 统计所有类型出现次数
genre_counter = Counter()
for genres_str in movies['GENRES'].dropna():
    for g in genres_str.split('/'):
        g = g.strip()
        if g:
            genre_counter[g] += 1

genres_sorted = genre_counter.most_common(20)
genre_names = [g for g, _ in genres_sorted]
genre_counts = [c for _, c in genres_sorted]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 类型频次
ax = axes[0]
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(genre_names)))
bars = ax.barh(range(len(genre_names)), genre_counts, color=colors)
ax.set_yticks(range(len(genre_names)))
ax.set_yticklabels(genre_names)
ax.invert_yaxis()
ax.set_title('Movie Genre Distribution', fontsize=14)
ax.set_xlabel('Count')
for bar, count in zip(bars, genre_counts):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, str(count), va='center')

# 类型-评分箱线图
ax = axes[1]
top_genres = [g for g, c in genre_counter.most_common(8)]
genre_rating_data = []
genre_labels = []
for genre in top_genres:
    genre_movies = movies[movies['GENRES'].str.contains(genre, na=False)].index
    genre_ratings = ratings[ratings['movieId'].isin(genre_movies)]['rating']
    genre_rating_data.append(genre_ratings.values)
    genre_labels.append(genre)

bp = ax.boxplot(genre_rating_data, labels=genre_labels, patch_artist=True)
for patch, c in zip(bp['boxes'], colors[:len(top_genres)]):
    patch.set_facecolor(c)
ax.set_title('Rating Distribution by Genre', fontsize=14)
ax.set_xlabel('Genre')
ax.set_ylabel('Rating')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. 年份分布

In [ ]:
valid_years = movies['YEAR'].dropna()
valid_years = valid_years[valid_years.between(1900, 2030)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.hist(valid_years, bins=30, color='teal', edgecolor='white')
ax.set_title(f'Movie Year Distribution ({int(valid_years.min())}-{int(valid_years.max())})', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Count')

ax = axes[1]
# 每年平均评分
year_ratings = []
for year in sorted(valid_years.dropna().unique()):
    if pd.isna(year):
        continue
    year_movies = movies[movies['YEAR'] == year].index
    year_rating = ratings[ratings['movieId'].isin(year_movies)]['rating']
    if len(year_rating) > 0:
        year_ratings.append({'year': int(year), 'mean_rating': year_rating.mean(), 'count': len(year_rating)})

yr_df = pd.DataFrame(year_ratings)
ax.scatter(yr_df['year'], yr_df['mean_rating'], s=yr_df['count']/10, alpha=0.6, color='coral')
ax.set_title('Average Rating by Year (size = #ratings)', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Mean Rating')
ax.axhline(ratings['rating'].mean(), color='gray', linestyle='--', alpha=0.5, label='Global Mean')
ax.legend()

plt.tight_layout()
plt.show()

## 6. 用户-电影交互矩阵稀疏度

In [ ]:
matrix, user_map, movie_map = build_user_item_matrix(ratings)

n_users = len(user_map)
n_movies = len(movie_map)
n_ratings = matrix.nnz
sparsity = 1.0 - n_ratings / (n_users * n_movies)

print(f'User-Item Matrix: {n_users} users x {n_movies} movies')
print(f'Non-zero entries: {n_ratings:,}')
print(f'Sparsity: {sparsity:.4%}')
print(f'Density:  {1-sparsity:.4%}')

In [ ]:
# 可视化交互矩阵（采样前100用户x50电影）
n_sample_users = min(100, n_users)
n_sample_movies = min(50, n_movies)
sample = matrix[:n_sample_users, :n_sample_movies].toarray()

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(sample, aspect='auto', cmap='YlOrRd', vmin=0, vmax=5)
ax.set_title(f'User-Item Rating Matrix (sample {n_sample_users} users x {n_sample_movies} movies)', fontsize=14)
ax.set_xlabel('Movie Index')
ax.set_ylabel('User Index')
plt.colorbar(im, ax=ax, label='Rating')
plt.tight_layout()
plt.show()

## 7. 数据清洗后的统计

In [ ]:
# 看看过滤稀疏前后的对比
filtered = filter_sparse(ratings, min_user_ratings=5, min_item_ratings=5)
print('Before filtering:')
print(f'  Ratings: {len(ratings):,}, Users: {ratings["userId"].nunique():,}, Movies: {ratings["movieId"].nunique():,}')
sp_before = len(ratings) / (ratings["userId"].nunique() * ratings["movieId"].nunique()) * 100
print(f'  Sparsity: {sp_before:.4f}%')

print()
print('After filtering (>=5 ratings per user, >=5 per movie):')
print(f'  Ratings: {len(filtered):,}, Users: {filtered["userId"].nunique():,}, Movies: {filtered["movieId"].nunique():,}')
sp_after = len(filtered) / (filtered["userId"].nunique() * filtered["movieId"].nunique()) * 100
print(f'  Sparsity: {sp_after:.4f}%')
print(f'  Ratings retained: {len(filtered)/len(ratings)*100:.1f}%')

## 总结

通过以上 EDA 可以发现：
1. **评分分布**: 用户评分集中在 3-5 分，豆瓣评分分布较均匀
2. **用户活跃度**: 大多数用户评分数较少（长尾分布），少数用户非常活跃
3. **电影流行度**: 存在明显的马太效应，头部电影占据了大量评分
4. **类型偏好**: 剧情、喜剧、爱情是最常见的电影类型
5. **数据稀疏**: 即使用过滤后的数据，user-item 矩阵仍然很稀疏，这对协同过滤模型是挑战
6. **建模建议**: 需要结合内容特征来缓解冷启动问题